<a href="https://colab.research.google.com/github/Krishnasri-kanduri/Collab/blob/QC-lab/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import random
def rand_bits(n): return [random.randint(0,1) for _ in range(n)]
def rand_bases(n): return [random.randint(0,1) for _ in range(n)] # 0=Z, 1=X
def bb84(n=100, eavesdrop=False, eaves_drop_rate=0.2):
    alice_bits = rand_bits(n)
    alice_bases = rand_bases(n)
    # Alice encodes; Bob chooses random measurement bases
    bob_bases = rand_bases(n)
    # Eve (optional) intercept/resend with random bases on a fraction of pulses
    eve_bases = rand_bases(n) if eavesdrop else [None]*n
    # Transmission outcomes
    measured = []
    for i in range(n):
        bit = alice_bits[i]
        base = alice_bases[i]
        # Eve step
        if eavesdrop and random.random() < eaves_drop_rate:
            eve_base = eve_bases[i]
            eve_result = bit if eve_base == base else random.randint(0,1)
            # Eve resends eve_result prepared in eve_base
            prep_bit, prep_base = eve_result, eve_base
        else:
            prep_bit, prep_base = bit, base
        # Bob measures
        bob_bit = prep_bit if bob_bases[i] == prep_base else random.randint(0,1)
        measured.append(bob_bit)
    # Sifting: keep positions where bases match
    sift = [i for i in range(n) if bob_bases[i] == alice_bases[i]]
    key_alice = [alice_bits[i] for i in sift]
    key_bob = [measured[i] for i in sift]
    # Estimate QBER on a small sample
    sample = min(20, len(sift))
    errs = sum(1 for i in range(sample) if key_alice[i]!=key_bob[i]) if sample>0 else 0
    qber = errs / sample if sample>0 else 0.0
    print(f"Sifted key length: {len(sift)} | QBER≈{qber:.2%}")
    print("Alice key (first 32):", key_alice[:32])
    print("Bob key (first 32):", key_bob[:32])
bb84(n=200, eavesdrop=True, eaves_drop_rate=0.3)

Sifted key length: 96 | QBER≈5.00%
Alice key (first 32): [0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1]
Bob key (first 32): [0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1]
